---
title: Simple moving average (SMA)
execute:
  enabled: true
---

`q.indicator.sma` calculates the arithmetic mean over a trailing window. It accepts one ordered `Series`, preserves its index, and returns missing values until the window is full.

This example compares 50-session averages for AAPL and the SPY benchmark over the latest five shared years in qrt's bundled data. Both assets are indexed to 100 so their trends and averages are comparable despite different price levels.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
prices.tail()

,AAPL,SPY
datetime,,
2026-07-17,333.739990,743.289978
2026-07-20,326.589996,742.090027
2026-07-21,327.739990,748.280029
2026-07-22,325.890015,747.409973
2026-07-23,321.660004,738.179993


## Calculate the indicator

Call `sma` separately for each instrument. The function deliberately does not infer symbol boundaries from a long-form table.

In [2]:
window = 50
averages = prices.apply(lambda series: q.indicator.sma(series, window))
averages.tail()

,AAPL,SPY
datetime,,
2026-07-17,302.632416,743.233398
2026-07-20,303.419308,743.436312
2026-07-21,304.230600,743.807910
2026-07-22,304.887400,744.041617
2026-07-23,305.467000,744.057212


## Compare with SPY

The close prices and moving averages use the same initial-price denominator. A value of 120 therefore means 20% above that instrument's first close in this five-year window.

In [3]:
base = prices.iloc[0]
comparison = pd.concat(
    [
        prices.div(base).mul(100).add_suffix(" close"),
        averages.div(base).mul(100).add_suffix(f" SMA {window}"),
    ],
    axis=1,
)
fig = q.plot.col(
    comparison,
    title=f"AAPL and SPY with {window}-session simple moving averages",
    ylabel="Indexed value (start = 100)",
)
fig.show(renderer="notebook_connected")